## Audit direction aware edges

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # assumes notebook is in notebooks/, project root is one level up
sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence


In [2]:
from evaluation.benchmark import generate_questions
import pandas as pd

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")
questions = generate_questions(df, n_samples=15, seed=42)
for q in questions:
    print(f"query:  {q.query!r}")
    print(f"answer: {q.correct_answer!r}  (relation: {q.source_relation})\n")

c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


query:  'Who is Doramin a spouse of?'
answer: 'his little motherly witch of a wife'  (relation: spouse_of)

query:  'Who is Adair a protector of?'
answer: 'assistant engineer'  (relation: protector_of)

query:  'Who is Aurelius an enemy of?'
answer: 'vortigern'  (relation: enemy_of)

query:  'Who is The school an enemy of?'
answer: 'the enemy'  (relation: enemy_of)

query:  'Who is Mrs. Hare the mother of?'
answer: 'richard'  (relation: parent_mother_of)

query:  'Who is my wife a spouse of?'
answer: 'i'  (relation: spouse_of)

query:  'Who is Kali a leader of?'
answer: 'wahimas'  (relation: leader_of)

query:  'Who is Mr Brooke a companion of?'
answer: 'us'  (relation: companion_of)

query:  'Who is Pringle a member of?'
answer: 'school'  (relation: member_of)

query:  'Who is Death Valley located in?'
answer: 'armagosa range'  (relation: located_in)

query:  'Who is Mrs. Mirvan a companion of?'
answer: 'lord orville'  (relation: companion_of)

query:  'Who is M.C.C. a rival of?'
answ

In [ ]:
# single-hop fix: does every generated question read
# unambiguously in the SAME direction as its own ground truth?
from evaluation.benchmark import generate_questions
from embedding.relation_text import relation_to_question, MANUAL_TEMPLATES

questions = generate_questions(df, n_samples=25, seed=42)

print(f"Generated {len(questions)} questions (candidate pool restricted to "
      f"{len(MANUAL_TEMPLATES)}/48 templated relation types)\n")
for q in questions[:25]:
    print(f"query:  {q.query!r}")
    print(f"answer: {q.correct_answer!r}  (relation: {q.source_relation})\n")

# Confirms scope-narrowing worked: no question should exist for a
# relation type without a verified template.
untemplated = [q for q in questions if relation_to_question("x", q.source_relation) is None]
print(f"Questions generated for untemplated relations (should be 0): {len(untemplated)}")

Generated 25 questions (candidate pool restricted to 31/48 templated relation types)

query:  'Who is Doramin a spouse of?'
answer: 'his little motherly witch of a wife'  (relation: spouse_of)

query:  'Who is Adair a protector of?'
answer: 'assistant engineer'  (relation: protector_of)

query:  'Who is Aurelius an enemy of?'
answer: 'vortigern'  (relation: enemy_of)

query:  'Who is The school an enemy of?'
answer: 'the enemy'  (relation: enemy_of)

query:  'Who is Mrs. Hare the mother of?'
answer: 'richard'  (relation: parent_mother_of)

query:  'Who is my wife a spouse of?'
answer: 'i'  (relation: spouse_of)

query:  'Who is Kali a leader of?'
answer: 'wahimas'  (relation: leader_of)

query:  'Who is Mr Brooke a companion of?'
answer: 'us'  (relation: companion_of)

query:  'Who is Pringle a member of?'
answer: 'school'  (relation: member_of)

query:  'Who is Death Valley located in?'
answer: 'armagosa range'  (relation: located_in)

query:  'Who is Mrs. Mirvan a companion of?'
answ

In [4]:
# regenerate the SAME three previously-confirmed-reversed
# cases (child_of/Will, protector_of/Taug, leader_of/King Arthur) and
# check the fix directly against the known-bad examples, not just new
# random samples.
from embedding.relation_text import relation_to_question

known_reversals = [
    ("Will", "child_of", "mrs. brand"),
    ("Taug", "protector_of", "teeka"),
    ("King Arthur", "leader_of", "dacia"),
]
for entity1, rel, correct_answer in known_reversals:
    q = relation_to_question(entity1, rel)
    print(f"{q!r}")
    print(f"  ground truth (entity2): {correct_answer!r}")
    print(f"  does the question's own English now point at entity2's role? (manual check)\n")

'Who is Will a child of?'
  ground truth (entity2): 'mrs. brand'
  does the question's own English now point at entity2's role? (manual check)

'Who is Taug a protector of?'
  ground truth (entity2): 'teeka'
  does the question's own English now point at entity2's role? (manual check)

'Who is King Arthur a leader of?'
  ground truth (entity2): 'dacia'
  does the question's own English now point at entity2's role? (manual check)



In [8]:
# n-hop chain fix: verify against real sampled paths, not
# just hand-picked examples, and confirm invalidated chains are being
# dropped rather than silently produced with a None phrase.
from evaluation.benchmark import find_n_hop_paths, _chain_phrase
import random
from graph.corpus import load_corpus


corpus = load_corpus("../data/graphs/corpus.pkl")

sample_book_id = "106"
graph = corpus[sample_book_id]

paths = find_n_hop_paths(graph, hops=2, max_samples=15)
valid, dropped = 0, 0
for p in paths:
    phrase = _chain_phrase(p["start"], p["relations"])
    if phrase is None:
        dropped += 1
        continue
    valid += 1
    print(f"{phrase!r}")
    print(f"  expected final answer: {p['end']!r}\n")

print(f"\n{valid} valid chains, {dropped} dropped (unverified relation in chain)")

'Who is the companion of of the companion of of taug?'
  expected final answer: 'younger apes'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the friend of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the friend of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is the companion of of the companion of of taug?'
  expected final answer: 'tantor'

'Who is th